<img src="https://upload.wikimedia.org/wikipedia/commons/3/35/Uba_fiuba_ingenieria_logo.png" width="300" align="center">



# **Analisis de Series de Tiempo II**

# **Clase 2, Ejercicio Feature Engineering**

Importamos lo necesario

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

Obtenemos el dataset

In [ ]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv",
    parse_dates=["Date"],
    index_col="Date"
)
df = df.rename(columns={"Temp": "y"})
df = df.asfreq("D")
df["y"] = df["y"].interpolate()

In [ ]:
print("Rango:", df.index.min().date(), "a", df.index.max().date())
print("Observaciones:", len(df))

Rango: 1981-01-01 a 1990-12-31
Observaciones: 3652


Calendar Features

In [ ]:
df["dia_semana"] = df.index.dayofweek
df["mes"] = df.index.month
df["dia_mes"] = df.index.day
df["trimestre"] = df.index.quarter
df["dia_anio"] = df.index.dayofyear
df["es_finde"] = (df.index.dayofweek >= 5).astype(int)

Encoding ciclico (estacionalidad anual, no semanal)

In [ ]:
dias_en_anio = 365.25
df["doy_sin"] = np.sin(2 * np.pi * df["dia_anio"] / dias_en_anio)
df["doy_cos"] = np.cos(2 * np.pi * df["dia_anio"] / dias_en_anio)
df["mes_sin"] = np.sin(2 * np.pi * df["mes"] / 12)
df["mes_cos"] = np.cos(2 * np.pi * df["mes"] / 12)

In [ ]:
features_indice = [
    "dia_semana", "mes", "dia_mes", "trimestre", "dia_anio", "es_finde",
    "doy_sin", "doy_cos", "mes_sin", "mes_cos",
]

Lags

In [ ]:
for lag in [1, 2, 3, 7]:
    df[f"lag_{lag}"] = df["y"].shift(lag)

In [ ]:
features_lags = ["lag_1", "lag_2", "lag_3", "lag_7"]

Rolling

In [ ]:
for w in [3, 7]:
    df[f"roll_mean_{w}"] = df["y"].rolling(w).mean().shift(1)
    df[f"roll_std_{w}"] = df["y"].rolling(w).std().shift(1)
    df[f"roll_max_{w}"] = df["y"].rolling(w).max().shift(1)
    df[f"roll_min_{w}"] = df["y"].rolling(w).min().shift(1)

features_rolling = [
    "roll_mean_3", "roll_std_3", "roll_max_3", "roll_min_3",
    "roll_mean_7", "roll_std_7", "roll_max_7", "roll_min_7",
]

Expanding

In [ ]:
df["exp_mean"] = df["y"].expanding().mean().shift(1)
df["exp_std"] = df["y"].expanding().std().shift(1)

In [ ]:
features_expanding = ["exp_mean", "exp_std"]

In [ ]:
features_full = features_indice + features_lags + features_rolling + features_expanding

Tras los lags/rolling quedan NaN al inicio de la serie, los descartamos

In [ ]:
df_model = df.dropna(subset=features_full + ["y"]).copy()

Split temporal: ultimos 365 dias = test

In [ ]:
n_test = 365
train = df_model.iloc[:-n_test]
test  = df_model.iloc[-n_test:]

In [ ]:
X_train_idx, X_test_idx = train[features_indice], test[features_indice]
X_train_full, X_test_full = train[features_full], test[features_full]
y_train, y_test = train["y"], test["y"]

In [ ]:
print(f"Train: {train.index.min().date()} a {train.index.max().date()} ({len(train)} obs)")
print(f"Test: {test.index.min().date()} a {test.index.max().date()} ({len(test)} obs)")

Train: 1981-01-08 a 1989-12-31 (3280 obs)
Test: 1990-01-01 a 1990-12-31 (365 obs)


Modelos

In [ ]:
gbm_idx = GradientBoostingRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=42
)
gbm_idx.fit(X_train_idx, y_train)
pred_gbm_idx = gbm_idx.predict(X_test_idx)

In [ ]:
gbm_full = GradientBoostingRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=42
)
gbm_full.fit(X_train_full, y_train)
pred_gbm_full = gbm_full.predict(X_test_full)

Naive estacional sobre la serie original

In [ ]:
pred_naive = df["y"].shift(365).reindex(test.index)

Métricas

In [ ]:
def metrics(y_true, y_pred, nombre):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f"{nombre:35s} | MAE: {mae:6.3f} | RMSE: {rmse:6.3f} | MAPE: {mape:6.2f}%")
    return {"modelo": nombre, "MAE": mae, "RMSE": rmse, "MAPE": mape}

Resultados

In [ ]:
res = []
res.append(metrics(y_test, pred_naive, "Naive estacional (t-365)"))
res.append(metrics(y_test, pred_gbm_idx, "GBM solo-indice"))
res.append(metrics(y_test, pred_gbm_full, "GBM full (indice+lags+rolling)"))

Naive estacional (t-365)            | MAE:  2.867 | RMSE:  3.651 | MAPE:  28.23%
GBM solo-indice                     | MAE:  2.054 | RMSE:  2.645 | MAPE:  20.33%
GBM full (indice+lags+rolling)      | MAE:  1.680 | RMSE:  2.160 | MAPE:  17.49%


In [ ]:
tabla = pd.DataFrame(res).set_index("modelo")
print(tabla.round(3).to_string())

                                  MAE   RMSE    MAPE
modelo                                              
Naive estacional (t-365)        2.867  3.651  28.231
GBM solo-indice                 2.054  2.645  20.327
GBM full (indice+lags+rolling)  1.680  2.160  17.490
